# 💧 AquaSense AI — 03: Machine Learning Model Training
**Project:** Intelligent Water Quality Assessment and Potability Prediction Using Explainable Machine Learning  

### Overview:
In this notebook, we train all 8 classification models:
1. Logistic Regression (Baseline)
2. Decision Tree
3. Random Forest
4. XGBoost
5. LightGBM
6. Support Vector Machine (SVM)
7. Multi-Layer Perceptron (MLP)
8. Soft Voting Ensemble (RF + XGB + LGBM)

We also perform 5-fold Stratified Cross-Validation and track parameters with MLflow.


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np

from src.data.loader import DataLoader
from src.data.preprocessor import WaterQualityPreprocessor
from src.models.trainer import ModelTrainer
from src.utils.logger import logger

print("Training modules loaded.")


## 1. Load Data & Apply Fitted Preprocessor


In [ ]:
dl = DataLoader(data_path="../data/raw/water_potability.csv")
df = dl.load()
X_train, X_test, y_train, y_test = dl.split(df, test_size=0.20, random_state=42)

preprocessor = WaterQualityPreprocessor.load("../models/preprocessor.pkl")
X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

X_train_bal, y_train_bal = preprocessor.apply_smote(X_train_proc, y_train)
print(f"Balanced training set: {X_train_bal.shape}, Test set: {X_test_proc.shape}")


## 2. 5-Fold Stratified Cross-Validation Benchmark


In [ ]:
trainer = ModelTrainer(random_state=42)
cv_results = trainer.cross_validate_all(X_train_bal, y_train_bal, cv=5)

df_cv = pd.DataFrame(cv_results).T
df_cv[['cv_accuracy_mean', 'cv_f1_mean', 'cv_roc_auc_mean', 'cv_mcc_mean']].sort_values(by='cv_mcc_mean', ascending=False)


## 3. Train All 8 Models & Serialize Pickles


In [ ]:
trained_models = trainer.train_all(X_train_bal, y_train_bal, use_mlflow=False)
saved_paths = trainer.save_all(trained_models, path="../models")
print("Models saved successfully to disk:")
for k, v in saved_paths.items():
    print(f"  - {k}: {v}")
